DeFi Protocol Safety Scoring Tool

A lightweight Python-based risk assessment framework for decentralized finance. Designed to run seamlessly in Google Colab, no heavy tools like Slither/Mythril



1. Introduction

The DeFi space is exciting but tricky. It's full of opportunities, but also packed with risks like wild price swings, liquidity drops and past hacks that still haunt some projects. This tool is my take on making sense of that chaos for a few big players: Aave, Uniswap and Compound.

Basically, it pulls together market stats and historical security info to give each protocol a quick safety score. The financial side looks at things like TVL scale and recent trends, while the security side digs into known exploits and how fresh their audits are. The end result is a clean PDF report with scores, charts and notes on where the data came from.

I kept it simple on purpose with no fancy blockchain security analysis tools like Slither, Mythril or endless dependencies, so anyone can run it in Colab seamlessly. It's not meant to be investment advice, just a practical way to spot relative risks. Think of it as a starting point for deeper dives.

Cell 1 – Setup & Constants

In [36]:
!pip install scikit-learn fpdf2 --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import re
import time

from sklearn.linear_model import LinearRegression
from fpdf import FPDF
from fpdf.enums import XPos, YPos
from io import BytesIO
from datetime import datetime, timedelta

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Setup complete.")

PROTOCOLS = {
    "aave": {
        "coingecko_id": "aave",
        "defillama_slug": "aave",
        "github_repo": "aave/aave-v3-core",
        "immunefi_url": "https://immunefi.com/bug-bounty/aave/information",
        "type": "lending"
    },
    "uniswap": {
        "coingecko_id": "uniswap",
        "defillama_slug": "uniswap",
        "github_repo": "Uniswap/v4-core",
        "immunefi_url": "https://cantina.xyz/bounties/f9df94db-c7b1-434b-bb06-d1360abdd1be",
        "type": "dex"
    },
    "compound": {
        "coingecko_id": "compound-governance-token",
        "defillama_slug": "compound-finance",
        "github_repo": "compound-finance/compound-protocol",
        "immunefi_url": "https://immunefi.com/bug-bounty/compoundfinance/information",
        "type": "lending"
    }
}

print(f"Configured {len(PROTOCOLS)} protocols: {', '.join(PROTOCOLS.keys())}")

Setup complete.
Configured 3 protocols: aave, uniswap, compound


2. Methodology

To build this, I focused on reliable, free data sources and straightforward calculations. No overcomplicating things with blockchain scanners or machine learning, just using APIs and basic math to keep it fast and explainable.


Data Collection (APIs & Public Sources)

Everything starts with grabbing fresh (or cached) data:
Market metrics: From CoinGecko's free API. This gives 7-day price momentum and the 30-day absolute price swing as a medium-term realized risk proxy. It's quick and covers token basics without needing keys.

TVL and trends: Pulled from DefiLlama's protocol endpoint. Gets the current total locked value across chains and the last 30 daily snapshots for spotting declines. If the API hiccups (which happens in Colab), it falls back to cached values and notes that in the report.

For security, it's all from public reports, with no live scanning. I hardcoded summaries based on checks from sites like Rekt.news for exploits, protocol docs for audits and Immunefi for bounties. Easy to update if needed.

One thing to note: APIs can be flaky, so the code will flag when it uses backups. That way, you know if scores are live or not.

Cell 2 – Data Fetching Functions

In [37]:
def fetch_market_data(protocol):
    config  = PROTOCOLS[protocol]
    url     = f"https://api.coingecko.com/api/v3/coins/{config['coingecko_id']}"
    headers = {'User-Agent': 'Mozilla/5.0'}                                 # reduces API blocks

    try:
        response = requests.get(url, headers=headers,timeout=15).json()
        if 'market_data' not in response:
            raise KeyError('market_data')

        market_data = response["market_data"]
        data = {
            "price_change_7d": market_data["price_change_percentage_7d_in_currency"]["usd"],
            "price_swing_30d": abs(market_data.get("price_change_percentage_30d_in_currency", {}).get("usd", 0)) / 100
        }
        data['source'] = 'live'
        return data

    except Exception as e:
        print(f"Market data fetch failed for {protocol}: {e}")
        # Feb 2026 updated fallbacks
        fallbacks = {
            "aave":     {"price_change_7d": -15.0, "price_swing_30d": 0.302},
            "uniswap":  {"price_change_7d": -11.3, "price_swing_30d": 0.352},
            "compound": {"price_change_7d": -20.6, "price_swing_30d": 0.356},
        }
        data = fallbacks[protocol]
        data['source'] = 'fallback'
        return data



def fetch_tvl_data(protocol):
    config  = PROTOCOLS[protocol]
    url     = f"https://api.llama.fi/protocol/{config['defillama_slug']}"
    headers = {'User-Agent': 'Mozilla/5.0'}

    try:
        response = requests.get(url, headers=headers, timeout=15).json()
        tvl_hist = response.get("tvl", [])
        tvl_history = [e["totalLiquidityUSD"] for e in tvl_hist][-30:]    # most recent 30 days
        chain_tvls = response.get("currentChainTvls", {})
        total_tvl = tvl_history[-1] if tvl_history else 0                 # matches chart source
        borrowed = sum(v for k, v in chain_tvls.items() if '-borrowed' in k.lower() or '-debt' in k.lower())     # borrowed is tracked separately
        chain_count = len(set(k.split('-')[0] for k in chain_tvls if '-borrowed' not in k.lower() and '-debt' not in k.lower()))
        supply_tvls = {k: v for k, v in chain_tvls.items() if '-borrowed' not in k.lower() and '-debt' not in k.lower() and v > 0}
        top_chain_pct = round(max(supply_tvls.values()) / sum(supply_tvls.values()) * 100, 1) if supply_tvls else 50.0

        # Attempt to get the 24h vol data
        volume_24h = None
        if config['type'] == 'dex':
            try:
                vol_url = f"https://api.llama.fi/summary/dexs/{config['defillama_slug']}?excludeTotalDataChart=true&dataType=dailyVolume"
                vol_resp = requests.get(vol_url, headers=headers, timeout=10).json()
                volume_24h = vol_resp.get("total24h")
            except Exception:
                pass
        if volume_24h is None:
            volume_24h = 1.37e9 if protocol == "uniswap" else 0

        data = {
            "total_tvl":  total_tvl,
            "tvl_history": tvl_history or [0] * 30,
            "borrowed":   borrowed,
            "chain_count": chain_count,
            "volume_24h": volume_24h,
            "top_chain_pct": top_chain_pct
        }
        data['source'] = 'live'
        return data

    except Exception as e:
        print(f"TVL fetch failed for {protocol}: {e}")
        fallbacks = {
            "aave":     {"total_tvl": 25.0e9, "tvl_history": [25.0e9]*30, "borrowed": 7.0e9, "chain_count": 24, "volume_24h": 0,      "top_chain_pct": 60.0},
            "uniswap":  {"total_tvl": 3.2e9,  "tvl_history": [3.2e9]*30,  "borrowed": 0,     "chain_count": 45, "volume_24h": 1.37e9, "top_chain_pct": 55.0},
            "compound": {"total_tvl": 1.4e9,  "tvl_history": [1.4e9]*30,  "borrowed": 0.4e9, "chain_count": 10, "volume_24h": 0,      "top_chain_pct": 68.0},
        }
        data = fallbacks[protocol]
        data['source'] = 'fallback'
        return data



def fetch_utilization_and_chains(protocol, tvl_data):
    config = PROTOCOLS[protocol]
    metric_name = "Utilization (Lending)" if config['type'] == 'lending' else ("Efficiency (DEX - Vol/TVL)" if config['type'] == 'dex' else "N/A")

    try:
        total_tvl = tvl_data['total_tvl']
        borrowed  = tvl_data['borrowed']
        chains    = tvl_data['chain_count']
        source    = tvl_data['source']

        if config['type'] == 'lending':
            util = (borrowed / (total_tvl + borrowed)) * 100 if (total_tvl + borrowed) > 0 else 0
        elif config['type'] == 'dex':
            volume_24h = tvl_data['volume_24h']
            util = min((volume_24h / total_tvl) * 100, 100) if total_tvl > 0 else 0     # Volume/TVL as efficiency proxy
        else:
            util = 0
    except Exception as e:
        print(f"Util/chains calc failed for {protocol}: {e}")
        fallbacks = {
            "aave":     {"util": 40.7, "chains": 24},
            "uniswap":  {"util": 19.9, "chains": 45},
            "compound": {"util": 27.6, "chains": 10},
        }
        util   = fallbacks[protocol]["util"]
        chains = fallbacks[protocol]["chains"]
        source = 'fallback'

    return {'utilization': util, 'chain_count': chains, 'source': source, 'metric_name': metric_name}

3. Risk Analysis (Financial & Historical Security)

The core is three risk angles, blended into one score.

Financial risk weighs current market health:
- Bigger TVL means more stability (min(total_tvl / 1e9, 10), capped at 10 to avoid over-weighting giants like Aave).
- 7d price momentum (price_change_7d / 5): short-term signal, normalised to avoid sudden swings dominating the score.
- 30d price swing penalty, medium-term instability signal. price_swing_30d × 2: absolute return with directional sign dropped, so it's a realized price risk proxy rather than a directional momentum signal.
- Trend penalty, negative slope from linear regression penalises impact of TVL fluctuations.


For historical security, it's about track record:
- Starts high at 10, then subtracts for any exploits.
- More penalties for old or few audits, or noted vulnerabilities, even fixed ones.
- Bonuses for audit count (effective_audits / 3, capped at 2.5), recency (2025+ preferred), active bug bounty and formal verification.
- Additional penalties for governance attacks, cross-protocol exposure (×0.5 per dependency) and open issues count, a small proxy for unresolved technical debt.


For operational risk, it covers protocol resilience and infrastructure quality:
- Emergency pause capability, oracle dependency level, bridge exposure and chain concentration are hardcoded from protocol research.
- Development activity (commits on core repo in last 90 days) and chain concentration (top percentage of TVL) are fetched live from GitHub and DefiLlama respectively.


It relies on public info, so it may miss unreported issues. But it's transparent and based on verifiable events.

Data Processing & Visualization

Once data's in, processing is basic: linear regression for trends (sklearn), heatmaps/tables/charts with seaborn/matplotlib. Then fpdf assembles it into a PDF. Summaries first, visuals after.
I added source tracking (live vs fallback) to keep things honest and clear.

Cell 3 – Analysis (security history, trend prediction, scoring)

In [38]:
def fetch_bounty_and_code_age(protocol):
    config = PROTOCOLS[protocol]
    headers = {'User-Agent': 'Mozilla/5.0'}

    # Some websites (like Immunefi and Cantina) need a special way to load their content, which makes it hard to get live bounty figures reliably.
    # Therefore, the bounty sizes are pre-recorded as of Feb 2026.

    BOUNTY_SIZES = {
        "aave":     1_000_000,
        "uniswap":  15_500_000,
        "compound": 1_000_000,
    }
    bounty_size = BOUNTY_SIZES[protocol]

    # 'code age' is retrieved live from GitHub.
    try:
        base_url = f"https://api.github.com/repos/{config['github_repo']}"

        # Latest commit code age
        latest = requests.get(f"{base_url}/commits?per_page=1", headers=headers, timeout=10).json()
        last_date = latest[0]['commit']['author']['date'].split('T')[0]
        code_age_days = (datetime.now() - datetime.strptime(last_date, "%Y-%m-%d")).days

        # Commits in last 90 days active development signal (capped at 100 per page, sufficient)
        since = (datetime.now() - timedelta(days=90)).strftime("%Y-%m-%dT00:00:00Z")
        recent = requests.get(f"{base_url}/commits?since={since}&per_page=100", headers=headers, timeout=15).json()
        recent_commits_90d = len(recent) if isinstance(recent, list) else 0

        # Repo metadata open_issues_count (includes PRs, rough technical-debt proxy, low weight in scoring)
        repo_meta = requests.get(base_url, headers=headers, timeout=10).json()
        open_issues_count = repo_meta.get('open_issues_count', 0)

        return {
            'bounty_size': bounty_size,
            'code_age_days': code_age_days,
            'recent_commits_90d': recent_commits_90d,
            'open_issues_count':  open_issues_count,
            'source': 'live'
        }

    except Exception as e:
        print(f"GitHub fetch failed for {protocol}: {e}")
        fallbacks = {
            "aave":     {'code_age_days': 30, 'recent_commits_90d': 50, 'open_issues_count': 30},
            "uniswap":  {'code_age_days': 15, 'recent_commits_90d': 40, 'open_issues_count': 20},
            "compound": {'code_age_days': 1335, 'recent_commits_90d': 3,  'open_issues_count': 25},
        }
        return {'bounty_size': bounty_size, **fallbacks[protocol], 'source': 'fallback'}



def check_security_history(protocol, bounty_code_data):

    # 2026 data from Rekt.news, protocol docs and Immunefi
    history = {
        "aave": {
            "exploit_count": 0,
            "recent_vuln": 2,                      # 2023 mitigated, 2025 periphery (no loss)
            "audit_count": 12,                     # V3.6 2025 (5), V4 2026 (Sherlock no findings + final)
            "latest_audit_year": 2026,
            "issues": ["2023 vulnerability mitigated, no loss", "2025 periphery hack $51K not core"],
            "has_audits": True,
            "audit_note": "V3.6 2025 (5 reports) - V4 2026 (Sherlock contest no findings, final audits)",
            "governance_attack_surface": "low",    # Guardian multisig can veto any proposal instantly
            "cross_protocol_exposure":   1,        # GHO curve pools + LayerZero cross-chain governance
            "has_formal_verification":   True,     # Certora used continuously on V3 core
        },
        "uniswap": {
            "exploit_count": 0,
            "recent_vuln": 1,                      # Hook vulns noted in V4 pre-deployment audits audits. SIR 2025 exploit using V3 as a price oracle which is not a Uniswap protocol vulnerability
            "audit_count": 10,                     # V4 2024 Trail of Bits, ABDK, Spearbit, Certora, OpenZeppelin + Cantina contest (5+ auditors, no critical findings)
            "latest_audit_year": 2024,
            "issues": ["No core exploits", "Hook vulnerabilities noted in V4 audits pre-deployment", "SIR 2025 exploit used V3 as a price oracle, which is not a Uniswap protocol vulnerability"],
            "has_audits": True,
            "audit_note": "V4 2024 (5+ in contest) - multiple firms, $15.5M bug bounty active",
            "governance_attack_surface": "medium", # 7-day timelock but a16z holds ~15% UNI. No veto mechanism
            "cross_protocol_exposure":   2,        # V3 is the dominant on-chain TWAP oracle for hundreds of protocols. V4 hooks execute third-party code inside the protocol perimeter
            "has_formal_verification":   True,     # Certora + Trail of Bits formal methods on V4
        },
        "compound": {
            "exploit_count": 1,                    # 2021 $147M distributor bug (forks like Onyx/Sonne exploited)
            "recent_vuln": 0,
            "audit_count": 11,                     # Pre-2023 (OpenZeppelin/Trail of Bits)
            "latest_audit_year": 2022,
            "issues": ["No new 2023-2026 core exploits", "2021 distributor bug $147M mitigated", "Forks (Onyx, Sonne) remain vulnerable to the same pattern"],
            "has_audits": True,
            "audit_note": "Pre-2023 (Trail of Bits, OpenZeppelin) - no recent audits found, $1M bounty active",
            "governance_attack_surface": "medium", # COMP heavily concentrated among early VCs. 2022 hostile proposal attempt (failed, but demonstrated surface)
            "cross_protocol_exposure":   1,        # Chainlink oracle dependency. Fork ecosystem re-exploits same pattern
            "has_formal_verification":   False,    # No Certora or K-framework verification in public audit history
        }
    }
    data = dict(history[protocol])
    data.update(bounty_code_data)

    base = 10
    # Audits pre-2023 discounted: protocol may have changed substantially since then
    effective_audits = float(data["audit_count"]) if data["latest_audit_year"] >= 2023 else data["audit_count"] * 0.4
    audit_bonus      = min(effective_audits / 3, 2.5)     # audits don't offset real exploits/vulnerabilities
    exploit_penalty  = data["exploit_count"] * 3
    vuln_penalty     = data["recent_vuln"] * 1.5

    bounty_bonus = 0.5 if data['bounty_size'] >= 1_000_000 else 0
    code_age_penalty = 1 if data['code_age_days'] > 365 else 0
    recency_penalty  = 0

    if data["latest_audit_year"] < 2026:
        recency_penalty = 0.5
    if data["latest_audit_year"] < 2025:
        recency_penalty = 1.5
    if data["latest_audit_year"] < 2023:
        recency_penalty = 3

    security = base - exploit_penalty - vuln_penalty - recency_penalty + audit_bonus + bounty_bonus - code_age_penalty
    gov_surface_penalties = {"low": 0, "medium": -0.5, "high": -1.5}
    security += gov_surface_penalties.get(data["governance_attack_surface"], 0)
    security -= data["cross_protocol_exposure"] * 0.5
    if not data["has_formal_verification"]:
        security -= 0.5
    security -= min(data.get("open_issues_count", 0) / 200, 0.5)  # open_issues_count includes PRs, a small-weight proxy for unresolved technical debt
    security = max(0, min(10, security))

    return {
        "exploit_count": data["exploit_count"],
        "issues":     data["issues"],
        "has_audits": data["has_audits"],
        "audit_note": data["audit_note"],
        "security": round(security, 2),
        "bounty_size": data["bounty_size"],
        "code_age_days": data["code_age_days"],
        "recent_commits_90d": data.get("recent_commits_90d", 20)
    }



def predict_tvl_trend(tvl_history):
    if len(tvl_history) < 2:
        return {"direction": "Unknown", "slope": 0}
    x = np.arange(len(tvl_history)).reshape(-1, 1)
    y = np.array(tvl_history)
    model = LinearRegression().fit(x, y)
    slope = model.coef_[0]
    baseline = np.mean(y)

    # pct_slope: the daily change as fraction of mean TVL. -0.001 ≈ -0.1% /day ≈ -3%/month
    pct_slope = (slope / baseline) if baseline > 0 else 0
    direction = "Declining" if pct_slope < -0.001 else "Stable/Increasing"
    return {"direction": direction, "slope": slope}



def compute_safety_scores(protocol, market, tvl, sec_data, trend, util_chains):
    config = PROTOCOLS[protocol]
    if not market or not tvl:
        return {"financial": 5, "security": 5, "operational": 5, "total": 5}

    price_trend = market.get("price_change_7d", 0)        # 7d momentum
    swing_penalty = market.get("price_swing_30d", 0) * 2  # 30d absolute price swing (directional sign dropped) - medium-term realized price risk proxy
    tvl_size_score = min(tvl["total_tvl"] / 1e9, 10)
    trend_penalty = -abs(trend["slope"]) / 5e8 if trend["slope"] < 0 else 0
    util_penalty = -1 if (config['type'] == 'lending' and util_chains['utilization'] > 80) or (config['type'] == 'dex' and util_chains['utilization'] < 10) else 0
    chain_penalty = -0.5 if util_chains['chain_count'] < 3 else 0

    financial = tvl_size_score + price_trend / 5 - swing_penalty + trend_penalty + util_penalty + chain_penalty
    financial = round(max(-10, min(financial, 10)), 2)    # Negative scores for very high-risk scenarios

    operational_score = compute_operational_risk(protocol, tvl, sec_data)

    security = sec_data["security"]

    total = round((financial + security + operational_score) / 3, 2)

    return {"financial": financial, "security": security, "operational": operational_score, "total": total}



def compute_operational_risk(protocol, tvl_data, sec_data):

    # Research-based (Feb 2026)
    op_profiles = {
        "aave": {
            "emergency_pause":  True,         # Guardian multisig can pause all markets instantly
            "oracle_dependency": "medium",    # Chainlink primary + Aave price sentinel fallback
            "bridge_exposure":  True,         # Cross-chain governance via LayerZero. GHO cross-chain minting
        },
        "uniswap": {
            "emergency_pause":  False,        # V4 core is immutable, no pause function exists
            "oracle_dependency": "none",      # AMM is the price discovery mechanism. No external oracle dependency for core operation
            "bridge_exposure":  False,        # Independent deployments per chain. No shared cross-chain state
        },
        "compound": {
            "emergency_pause":  True,         # Pause guardian exists for individual markets
            "oracle_dependency": "high",      # Chainlink only, with no fallback. Oracle failure directly halts liquidations
            "bridge_exposure":  True,         # Cross-chain deployments rely on bridges
        }
    }

    protocol_operational_profile = op_profiles[protocol]
    score = 10.0

    # Emergency pause: inability to pause on discovery of an exploit is an operational risk, even if immutability belongs in security
    if not protocol_operational_profile["emergency_pause"]:
        score -= 0.5

    # Oracle dependency: single-source oracle failure can freeze liquidations or enable price manipulation
    oracle_penalties = {"none": 0, "medium": -0.5, "high": -1.5}
    score += oracle_penalties[protocol_operational_profile["oracle_dependency"]]

    # Bridge exposure: each cross-chain integration is an additional vulnerability for attacks and failures
    if protocol_operational_profile["bridge_exposure"]:
        score -= 0.5

    # Chain concentration (live from DefiLlama currentChainTvls). High concentration on one chain can be a single-point-of-failure for TVL and liquidity
    top_chain_pct = tvl_data.get("top_chain_pct", 50.0)
    if top_chain_pct > 70:
        score -= 1.5
    elif top_chain_pct > 50:
        score -= 0.75

    # Development activity (live from GitHub, last 90 days). Near-zero commits on a live protocol with $1B+ TVL signals maintenance risk
    recent_commits = sec_data.get("recent_commits_90d", 20)
    if recent_commits < 5:
        score -= 1.5
    elif recent_commits < 20:
        score -= 0.5

    return round(max(0, min(10, score)), 2)

Cell 4 – Visualization & PDF generation

In [39]:
def plot_tvl_forecast(protocol, tvl_history):

    if len(tvl_history) < 2:
        return None

    x_historical_days = np.arange(-len(tvl_history) + 1, 1)
    y_historical_tvl  = np.array(tvl_history)
    model = LinearRegression().fit(x_historical_days.reshape(-1,1), y_historical_tvl)
    x_future_days = np.arange(1, 8)
    y_predicted_tvl = model.predict(x_future_days.reshape(-1,1))
    fig = plt.figure(figsize=(9, 4.5))
    plt.plot(x_historical_days, y_historical_tvl / 1e9, 'o-', label="Historical", color="blue")
    plt.plot(x_future_days, y_predicted_tvl / 1e9, '--', label="7-day linear trend", color="red")

    plt.title(f"{protocol.upper()} TVL Trend & Linear Projection ($B)")
    plt.xlabel("Days (0 = most recent)")
    plt.ylabel("TVL ($B)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig



def export_pdf_report(results):

    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)


    # Page 1: Per-protocol details
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, f"DeFi Protocol Safety Report ({datetime.now().strftime('%b %Y')})", new_x=XPos.LMARGIN, new_y=YPos.NEXT, align="C")
    pdf.set_font("Helvetica", "I", 8)
    pdf.multi_cell(0, 5, "Disclaimer: Basic risk tool using public data. Not financial advice or full audit.")
    pdf.ln(5)

    for proto, res in results.items():
        pdf.set_font("Helvetica", "B", 12)
        pdf.cell(0, 5, f"{proto.upper():<10} {res['total']:.2f}/10 -> {res['rating']}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font("Helvetica", size=10)
        pdf.cell(0, 5, f"Financial : {res['financial']:>6.2f}/10 | Security: {res['security']:>6.2f}/10 | Operational: {res['operational']:>5.2f}/10", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font("Helvetica", "I", 8)
        pdf.cell(0, 4, "Total = (Financial + Security + Operational) / 3", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.set_font("Helvetica", size=10)
        pdf.cell(0, 5, f"TVL       : $ {res['tvl']['total_tvl']/1e9:>5.1f}B | Trend: {res['trend']['direction']}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.cell(0, 5, f"{res['util_chains']['metric_name']}: {res['util_chains']['utilization']:.1f}% | Chains: {res['util_chains']['chain_count']}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.cell(0, 5, f"Data Source: {res['data_source']}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.cell(0, 5, f"Audited   : {'Yes' if res['sec']['has_audits'] else 'No'} | Exploits: {res['sec']['exploit_count']}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        audit_note = res['sec']['audit_note']
        bounty_size = res['sec']['bounty_size']
        code_age_days = res['sec']['code_age_days']
        if bounty_size >= 1_000_000:
            bounty_display = f"${bounty_size / 1_000_000:.1f}M"
        else:
            bounty_display = f"${bounty_size:,}"
        pdf.multi_cell(0, 5, f"Audit Note: {audit_note} | Bounty: {bounty_display} | Code Age: {code_age_days} days", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        issues_str = '; '.join(res['sec']['issues']) or "None major"
        pdf.multi_cell(0, 5, f"Issues    : {issues_str}", new_x=XPos.LMARGIN, new_y=YPos.NEXT)
        pdf.ln(3)

    # Summary table
    pdf.ln(15)
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "Overall Protocol Summary", new_x=XPos.LMARGIN, new_y=YPos.NEXT, align="C")
    pdf.ln(5)
    col_widths = [40, 20, 20, 20, 20, 30, 20, 20]
    pdf.set_font("Helvetica", "B", 10)
    headers = ['Protocol', 'Total Score', 'Financial', 'Security', 'TVL(B)', 'Trend', 'Audit', 'Exploits']

    for i, header in enumerate(headers):
        pdf.cell(col_widths[i], 7, header, border=1, align='C')   # centering headers with borders
    pdf.ln()
    pdf.set_font("Helvetica", size=9)
    aligns = ['L', 'R', 'R', 'R', 'R', 'L', 'C', 'R']             # Alignments in table cells
    for proto, res in results.items():
        row_data = [
            proto.upper(),
            f"{res['total']:.2f}/10.00",
            f"{res['financial']:.2f}",
            f"{res['security']:.2f}",
            f"${res['tvl']['total_tvl']/1e9:.1f}",
            res['trend']['direction'],
            'Yes' if res['sec']['has_audits'] else 'No',
            str(res['sec']['exploit_count'])
        ]
        for i, item in enumerate(row_data):
            pdf.cell(col_widths[i], 6, item, border=1, align=aligns[i])
        pdf.ln()


    # Page 2: Heatmap
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "Safety Heatmap (Higher = Safer)", new_x=XPos.LMARGIN, new_y=YPos.NEXT, align="C")
    pdf.ln(5)

    data = pd.DataFrame({
        "Protocol": [p.upper() for p in results],
        "Financial": [results[p]["financial"] for p in results],
        "Security": [results[p]["security"] for p in results],
        "Operational": [results[p]["operational"] for p in results],
        "Total": [results[p]["total"] for p in results]
    }).set_index("Protocol")

    fig, ax = plt.subplots(figsize=(8,4))
    sns.heatmap(data.T, annot=True, cmap="RdYlGn", center=5, fmt=".1f", cbar_kws={'label': 'Score'})  # Center=5, yellow for better representation of risk on a 0-10 scale
    ax.set_xlabel('')                       # stop index name "Protocol" from leaking into the x-axis label
    plt.title("Component Scores")
    buf = BytesIO()
    plt.savefig(buf, format="png", dpi=200, bbox_inches="tight")
    plt.close()
    buf.seek(0)
    pdf.image(buf, x=15, w=180)

    # Footer if a fallback is used
    if any('fallback' in res['data_source'] for res in results.values()):
        pdf.set_font("Helvetica", "I", 8)
        footer_height = 10
        y_bottom_of_page = pdf.h - pdf.b_margin - footer_height
        pdf.set_y(y_bottom_of_page)
        pdf.cell(0, footer_height, "Note: Fallback data from Feb 2026 cache used for some metrics.", align="C")


    # Pages 3–5: TVL Charts
    for proto, res in results.items():
        pdf.add_page()
        pdf.set_font("Helvetica", "B", 14)
        pdf.cell(0, 10, f"TVL Trend - {proto.upper()}", new_x=XPos.LMARGIN, new_y=YPos.NEXT, align="C")
        pdf.ln(5)
        fig = plot_tvl_forecast(proto, res["tvl"]["tvl_history"])
        if fig:
            buf = BytesIO()
            fig.savefig(buf, format="png", dpi=200, bbox_inches="tight")
            plt.close(fig)
            buf.seek(0)
            pdf.image(buf, x=15, w=180)
        else:
            pdf.cell(0, 10, "Not enough data for chart", align="C")

    filename = "DeFi_Safety_Report.pdf"
    pdf.output(filename)
    print(f"PDF saved: {filename}")

    if IN_COLAB:
        files.download(filename)
    return filename

Cell 5 – Run everything cell loop

In [40]:
results = {}

for proto in PROTOCOLS:
    print(f"\nAnalyzing {proto.upper()}")
    market = fetch_market_data(proto)
    tvl = fetch_tvl_data(proto)
    util_chains = fetch_utilization_and_chains(proto, tvl)
    bounty_code = fetch_bounty_and_code_age(proto)
    sec = check_security_history(proto, bounty_code)
    trend = predict_tvl_trend(tvl["tvl_history"])
    scores = compute_safety_scores(proto, market, tvl, sec, trend, util_chains)

    if scores["total"] > 7:
        rating = "LOW RISK"
    elif scores["total"] > 4:
        rating = "MEDIUM RISK"
    else:
        rating = "HIGH RISK"

    source_labels = {
        'market': market.get('source', 'unknown'),
        'tvl' : tvl.get('source', 'unknown'),
        'bounty': bounty_code.get('source', 'unknown')
        }

    failed = [k for k, v in source_labels.items() if v != 'live']
    data_source = 'Live API' if not failed else f"Partial fallback ({', '.join(failed)}, static Feb 2026 values)"

    results[proto] = {
        "total": scores["total"],
        "financial": scores["financial"],
        "security": scores["security"],
        "operational": scores["operational"],
        "rating": rating,
        "tvl": tvl,
        "trend": trend,
        "sec": sec,
        "data_source": data_source,
        "util_chains": util_chains
    }
    time.sleep(1.5)                                                # CoinGecko free tier is about 30 requests/min

export_pdf_report(results)


Analyzing AAVE

Analyzing UNISWAP

Analyzing COMPOUND
PDF saved: DeFi_Safety_Report.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'DeFi_Safety_Report.pdf'

4. Past Results & Analysis

Running the tool produces a new PDF, resembling the screenshots in the screenshots folder in the repo, which are also called below.
Actual numbers shift with market data and also depend on live API data vs fallback.

Example-Run Results Showcase

In [41]:
# Important!!!

# The following results are not the most recent run output of the code. This cell showcases the uploaded screenshots from the screenshots folder in the repo, produced on a previous date.
# The output from your most recent run of the code is the PDF file downloaded on your system.


from IPython.display import Image

# Display the report summary page 1
github_image_url_1 = 'https://raw.githubusercontent.com/GeorgeKGM2058/DeFi-Protocol-Safety-Scoring-Tool/refs/heads/main/screenshots/report-summary-page1.png'
print("Report Summary - Page 1:")
display(Image(url=github_image_url_1))

# Display the report heatmap page 2
github_image_url_2 = 'https://raw.githubusercontent.com/GeorgeKGM2058/DeFi-Protocol-Safety-Scoring-Tool/refs/heads/main/screenshots/report-heatmap-page2.png'
print("\nReport Heatmap - Page 2:")
display(Image(url=github_image_url_2))

# Display the report TVL Aave page 3
github_image_url_3 = 'https://raw.githubusercontent.com/GeorgeKGM2058/DeFi-Protocol-Safety-Scoring-Tool/refs/heads/main/screenshots/report-tvl-aave-page3.png'
print("\nReport TVL Aave - Page 3:")
display(Image(url=github_image_url_3))

Report Summary - Page 1:



Report Heatmap - Page 2:



Report TVL Aave - Page 3:


From that specific run:

- Aave leads with a 7.93 with low risk. The 25.4bln USD TVL anchors the financial score, 2026 audits on both V3.6 and V4 (Sherlock contest, no findings) push security to 8.26 and the operational score of 7.50 reflects Ethereum chain concentration (>70% TVL) partially offset by high development activity.
- Uniswap sits at 5.46 with medium risk. Its smaller TVL at about 3.2B USD limits the financial score despite the solid price momentum. Security is at 8.13 due to the 15.5M USD bounty and comprehensive V4 2024 audits. Operational at 7.25 reflects the absence of a pause function and low v4-core commit activity post-launch stabilisation.
- Compound trails at 3.31, a higher risk. The 2021 147mln USD distributor exploit, pre-2023 audit history and near-zero development activity pull security to 3.06. Operational risk is at 5.75 due to < 5 commits in 90 days and Chainlink-only oracle with no fallback.

The heatmap captures the divergence: Aave is uniformly green, while Uniswap's and Compound's financial cells are clearly red. The primary risk driver here is audit staleness combined with exploit history, not current market conditions.

TVL charts show 30-day historical windows with linear projections. Note that a linear extrapolation on a 30-day DeFi TVL series is illustrative only. It captures the prevailing trend direction but does not model the non-linear dynamics (liquidation cascades, governance events, chain migrations) that dominate tail risk in this asset class.

Limitations: The security module is based on publicly reported events only. Unreported vulnerabilities and live governance attack surfaces are only partially captured. Governance concentration and cross-protocol exposure are included as penalty signals, with the inputs being hardcoded from research. The financial score incorporates 7-day governance token price momentum, which reflects market sentiment rather than protocol solvency. The 30-day window feeds the instability penalty separately. TVL and utilization are the primary protocol-health anchors. So, a protocol with strong TVL and weak token price will score below reality. This is a deliberate design tradeoff and not a data error.

5. Conclusion

The tool is intentionally narrow: three protocols, public data only, no on-chain scanning. That scope makes it auditable, reproducible, fast and easy to run online (Google Colab), which matters more for this use case than pure coverage.

References


CoinGecko API docs (2026): https://www.coingecko.com/en/api

DefiLlama protocol data: https://defillama.com/docs/api

Rekt.news exploit database: https://rekt.news

Immunefi bounty listings: https://immunefi.com

Aave developer docs & audits: https://docs.aave.com

Uniswap governance & security: https://uniswap.org

Compound finance resources: https://compound.finance/docs